In [3]:
import pandas as pd

df = pd.read_csv("../data/ideologia_congreso_chile_2002_2026_wide_format.csv")
df_2022 = df[~df['dim_ideology_P6'].isna()]
features = ["Id", "Name", "dim_ideology_P6", "party_1_P6"]
df_2022 = df_2022[features].reset_index(drop=True)
df_2022.head()
del df, features

In [5]:
df_2022['party_1_P6'].value_counts()

party_1_P6
IND        37
RN         25
UDI        21
PS         14
PC         11
PREP       10
DC          8
RD          6
PDG         5
PR          2
PCS         2
EVOP        2
PPD         2
PH          2
PEV         2
LIBERAL     2
COMUNES     1
IGUAL       1
PL          1
FRVS        1
PSC         1
PRI         1
Name: count, dtype: int64

## Obtener biografías


In [94]:
links = {}
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
for i in range(1,6):
    url = f"https://www.bcn.cl/historiapolitica/resenas_parlamentarias/index.html?categ=en_ejercicio&filtros=1&pagina={i}&K=1#listado_parlamentarios"
    print(url)
    resp = requests.get(url, headers=headers)
    soup = BeautifulSoup(resp.text, 'html.parser')
    parlamentarios = soup.find('div', id="contenedorResultados").find_all('a')
    print(f"Número de parlamentrarios/as encontrados en esta página: {len(parlamentarios)}")
    for p in parlamentarios:
        links[p.text.strip()] = "https://www.bcn.cl" + p['href']

https://www.bcn.cl/historiapolitica/resenas_parlamentarias/index.html?categ=en_ejercicio&filtros=1&pagina=1&K=1#listado_parlamentarios
Número de parlamentrarios/as encontrados en esta página: 44
https://www.bcn.cl/historiapolitica/resenas_parlamentarias/index.html?categ=en_ejercicio&filtros=1&pagina=2&K=1#listado_parlamentarios
Número de parlamentrarios/as encontrados en esta página: 44
https://www.bcn.cl/historiapolitica/resenas_parlamentarias/index.html?categ=en_ejercicio&filtros=1&pagina=3&K=1#listado_parlamentarios
Número de parlamentrarios/as encontrados en esta página: 44
https://www.bcn.cl/historiapolitica/resenas_parlamentarias/index.html?categ=en_ejercicio&filtros=1&pagina=4&K=1#listado_parlamentarios
Número de parlamentrarios/as encontrados en esta página: 44
https://www.bcn.cl/historiapolitica/resenas_parlamentarias/index.html?categ=en_ejercicio&filtros=1&pagina=5&K=1#listado_parlamentarios
Número de parlamentrarios/as encontrados en esta página: 29
Error in callback <bound 

In [97]:
import pickle
with open("../data/bcn_links.pkl", "wb") as f:
    pickle.dump(links, f)

In [99]:
from rapidfuzz import process, fuzz

# Esta lista la deberías obtener una sola vez (scraping del índice de la BCN)
nombres_reales_bcn = list(links.keys())

def encontrar_nombre_real(mi_nombre, lista_bcn):
    # 'token_set_ratio' es genial porque ignora si falta un nombre intermedio
    # o si el orden es distinto.
    resultado = process.extractOne(
        mi_nombre, 
        lista_bcn, 
        scorer=fuzz.token_set_ratio
    )
    
    nombre_match, score, index = resultado
    
    # Un score > 85 suele ser una coincidencia segura
    if score > 75:
        return nombre_match
    print(f"⚠️ No se encontró una coincidencia clara para '{mi_nombre}'. Mejor resultado: '{nombre_match}' con score {score}.")
    return None

nombres_reales = []
for nombre in df_2022['Name']:
    nombre_real = encontrar_nombre_real(nombre, nombres_reales_bcn)
    nombres_reales.append(nombre_real)

df_2022['Name_BCN'] = pd.Series(nombres_reales)

⚠️ No se encontró una coincidencia clara para 'Mercedes Bulnes Núñez'. Mejor resultado: 'Velásquez Núñez Esteban Jorge' con score 48.0.


In [101]:
df_2022 = df_2022[~df_2022['Name_BCN'].isna()]

In [6]:
import asyncio
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
from tqdm import tqdm

async def obtener_biografia(nombre: str, retries=2):
    url = links.get(nombre)
    
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        # Usamos un contexto para simular mejor un navegador
        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0 Safari/537.36"
        )
        page = await context.new_page()
        
        content = None
        for intento in range(retries + 1):
            try:
                # 1. Intentamos cargar con un tiempo de espera razonable
                await page.goto(url, timeout=45000, wait_until="domcontentloaded")
                
                # 2. Esperamos un poco más por el selector
                selector_objetivo = 'div[itemtype="http://schema.org/WebPage"]'
                await page.wait_for_selector(selector_objetivo, timeout=20000)
                
                html_renderizado = await page.content()
                soup = BeautifulSoup(html_renderizado, 'html.parser')
                content = soup.find('div', itemtype="http://schema.org/WebPage")
                
                if content:
                    break # Si lo encontramos, salimos del bucle de reintentos
                    
            except Exception as e:
                if intento < retries:
                    print(f"Reintentando {nombre} (Intento {intento + 1})...")
                    await asyncio.sleep(3) # Esperamos antes de reintentar
                else:
                    print(f"Falló definitivamente {nombre} tras {retries} reintentos: {e}")
        
        await browser.close()
        return content
# Para llamarlo en un script normal:
# resultado = asyncio.run(obtener_biografia("Nombre"))

# Si estás en un Notebook o Colab, simplemente usa:
# resultado = await obtener_biografia("Nombre")

biografias = []
for nombre in tqdm(df_2022['Name_BCN'], desc="Obteniendo biografías"):
    biografia = await obtener_biografia(nombre)
    biografias.append(biografia)

Obteniendo biografías:  20%|█▉        | 31/156 [03:28<13:21,  6.41s/it]

Reintentando González Gatica Félix Marcelo (Intento 1)...
Reintentando González Gatica Félix Marcelo (Intento 2)...


Obteniendo biografías:  21%|██        | 32/156 [04:37<51:45, 25.04s/it]

Falló definitivamente González Gatica Félix Marcelo tras 2 reintentos: Page.wait_for_selector: Timeout 20000ms exceeded.
Call log:
  - waiting for locator("div[itemtype=\"http://schema.org/WebPage\"]") to be visible

Reintentando Hertz Cádiz Carmen (Intento 1)...
Reintentando Hertz Cádiz Carmen (Intento 2)...


Obteniendo biografías:  21%|██        | 33/156 [05:45<1:18:10, 38.14s/it]

Falló definitivamente Hertz Cádiz Carmen tras 2 reintentos: Page.wait_for_selector: Timeout 20000ms exceeded.
Call log:
  - waiting for locator("div[itemtype=\"http://schema.org/WebPage\"]") to be visible



... (25 outputs left out)

Reintentando Sánchez Ossa Luis Fernando (Intento 1)...
Reintentando Sánchez Ossa Luis Fernando (Intento 2)...


Obteniendo biografías:  90%|█████████ | 141/156 [27:18<06:27, 25.87s/it]

Falló definitivamente Sánchez Ossa Luis Fernando tras 2 reintentos: Page.wait_for_selector: Timeout 20000ms exceeded.
Call log:
  - waiting for locator("div[itemtype=\"http://schema.org/WebPage\"]") to be visible



Obteniendo biografías: 100%|██████████| 156/156 [29:01<00:00, 11.16s/it]


In [10]:
# import pickle
# with open("../data/bcn_biografias_2022.pkl", "wb") as f:
#     pickle.dump(biografias, f)

# otra opcion
bio_sr = pd.Series(biografias)
bio_sr.to_csv("../data/bcn_biografias_2022.csv", index=False)

In [11]:
for i in range(len(biografias)):
    if biografias[i] is None:
        print(f"Biografía faltante para: {df_2022['Name_BCN'].iloc[i]}")

Biografía faltante para: González Gatica Félix Marcelo
Biografía faltante para: Hertz Cádiz Carmen
Biografía faltante para: Labbé Martínez Cristián Andrés
Biografía faltante para: Barchiesi Chávez Chiara Natalia
Biografía faltante para: Gazmuri Vieira Ana María
Biografía faltante para: Leal Bizama Henry
Biografía faltante para: Pulgar Castillo Juan Francisco
Biografía faltante para: Sánchez Ossa Luis Fernando


In [14]:
for i in range(len(biografias)):
    if biografias[i] is None:
        name = df_2022['Name_BCN'].iloc[i]
        biografias[i] = await obtener_biografia(name, retries=5)

In [15]:
for i in range(len(biografias)):
    if biografias[i] is None:
        print(f"Biografía faltante para: {df_2022['Name_BCN'].iloc[i]}")

In [16]:
df_2022['html_biografia'] = pd.Series(biografias)
df_2022.to_csv("../data/congreso_2022_2026_biografias.csv", index=False)

## Convertir biografías en resumenes factuales neutros

Una lista de puntos describiendo al diputado y sus características (partido, estudios, gestiones). 

No debe incluir apreciaciones o declaraciones subjetivas para no filtrar esta info y sesgar los probes

In [1]:
import pickle
with open("../data/bcn_links.pkl", "rb") as f:
    links = pickle.load(f)

import pandas as pd
df_2022 = pd.read_csv("../data/congreso_2022_2026_biografias.csv")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from tqdm import tqdm
import os

# Configuración del modelo local
model_id = "mistralai/Mistral-7B-Instruct-v0.1" # Puedes cambiarlo por Llama-2-7b-chat-hf u otro
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    torch_dtype=torch.float16, 
    device_map="auto",
    low_cpu_mem_usage=True
)

def generar_resumen(nombre, partido, texto_biografia):
    prompt = f"""USER: A continuación te presento una biografía del diputado/a chileno/a {nombre} del partido {partido}. 
Resúmela en una lista corta de puntos con características factuales (profesión, partido, gestiones principales). 
Es crucial que no contenga apreciaciones subjetivas, adjetivos calificativos ni juicios de valor.

Biografía:
{texto_biografia[:3000]} # Limitamos longitud para evitar problemas de contexto

ASSISTANT: Aquí tienes el resumen factual:
"""
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        output = model.generate(
            **inputs, 
            max_new_tokens=250, 
            temperature=0.1, 
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    
    response = tokenizer.decode(output[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True)
    return response.strip()

resumenes = []

for index, row in tqdm(df_2022.iterrows(), total=len(df_2022)):
    nombre = row['Name']
    partido = row['party_1_P6']
    biografia_html = row['html_biografia']
    
    # Extraer texto limpio del HTML si es necesario
    if hasattr(biografia_html, 'get_text'):
        biografia_texto = biografia_html.get_text(separator=' ', strip=True)
    else:
        # Si viene como string de un CSV anterior
        from bs4 import BeautifulSoup
        biografia_texto = BeautifulSoup(str(biografia_html), 'html.parser').get_text(separator=' ', strip=True)

    try:
        resumen = generar_resumen(nombre, partido, biografia_texto)
    except Exception as e:
        print(f"Error procesando a {nombre}: {e}")
        resumen = None
        
    resumenes.append(resumen)

df_2022["resumen"] = resumenes
df_2022.to_csv("../data/congreso_2022_2026_con_resumen.csv", index=False)

# Alternativa a resumenes: Obtener intervenciones de los diputados

En vez de promptear con resumen factual del diputado/a, pasar intervenciones que ha hecho. De todas formas deberían activar las neuronas que guardan el espacio político del modelo.

In [ ]:
from bs4 import BeautifulSoup
from tqdm import tqdm
import requests

def get_intervenciones(html):
    html = row['html_biografia']
    tabla_intervenciones = html.find('table', id='tabla_participaciones')
    if tabla_intervenciones is None:
        return None
    body = tabla_intervenciones.find('tbody')
    filas = body.find_all('tr')
    intervenciones = []
    for fila in filas:
        link = fila.find('a')
        href = link['href']

        resp = requests.get(href)
        if not resp: 
            return None
        resp.encoding = 'utf-8'
        intervencion_html = BeautifulSoup(resp.text, 'html.parser')
        contenido = intervencion_html.find('debatesection', {'name':"Participacion"})
        if not contenido:
            continue
        texto_contenido = contenido.find_all('p')
        texto_final = ""
        for p in texto_contenido[6:]:
            texto_final += p.get_text(separator=' ', strip=True) + "\n"
        intervenciones.append(texto_final)
    if len(intervenciones) <= 2:
        print(f"⚠️ Solo se encontraron {len(intervenciones)} intervenciones para {row['Name']}.")
    return intervenciones

intervenciones_list = []
for i, row in tqdm(df_2022.iterrows(), desc="Extrayendo intervenciones:", total=len(df_2022)):

    intervenciones_i = get_intervenciones(row['html_biografia'])

    intervenciones_list.append(intervenciones_i)


Extrayendo intervenciones::  53%|█████▎    | 82/156 [01:13<00:41,  1.77it/s]/home/matias/Documentos/chilean-politic-linear-representations/.venv/lib/python3.12/site-packages/bs4/builder/__init__.py:409: RuntimeWarning: coroutine 'obtener_biografia' was never awaited
  universal: Set[str] = self.cdata_list_attributes.get("*", set())
Extrayendo intervenciones:: 100%|██████████| 156/156 [02:21<00:00,  1.10it/s]


In [155]:
df_2022['intervenciones'] = pd.Series(intervenciones_list)
df_2022.to_csv('../data/congreso_2022_2026_biografias.csv', index=False)

In [165]:
df_2022.head()

,Id,Name,dim_ideology_P6,party_1_P6,Name_BCN,html_biografia,intervenciones
0,74,Jaime Naranjo Ortiz,-1.100019,PS,Naranjo Ortiz Jaime,"[\n, [\n, [\n, h3>Ficha parlamentaria</h3, \n,...","[Señor Presidente, qué duda cabe de que el Pla..."
1,803,René Alinco Bustos,-0.561267,IND,Alinco Bustos René,"[\n, [\n, [\n, h3>Ficha parlamentaria</h3, \n,...","[Señor Presidente, por su intermedio, solicito..."
2,815,Sergio Bobadilla Muñoz,0.646541,UDI,Bobadilla Muñoz Sergio Enrique,"[\n, [\n, [\n, h3>Ficha parlamentaria</h3, \n,...","[Señor Presidente, lamento que no esté el comp..."
3,872,Jaime Mulet Martínez,-0.473463,DC,Mulet Martínez Jaime,"[\n, [\n, [\n, h3>Ficha parlamentaria</h3, \n,...","[Sin duda, la transición energética es necesar..."
4,917,Gastón Von Mühlenbrock Zamora,0.720369,UDI,Von Mühlenbrock Zamora Gastón,"[\n, [\n, [\n, h3>Ficha parlamentaria</h3, \n,...","[Señor Presidente, me presento hoy ante ustede..."


# Probing and Intervention

In [ ]:
import torch
import numpy as np
import pandas as pd
import pickle
import os
from tqdm import tqdm
from baukit import TraceDict
from einops import rearrange
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold
from scipy.stats import spearmanr
from transformers import AutoModelForCausalLM, AutoTokenizer

os.makedirs('./results_replication', exist_ok=True)

# Probing Partisan Slant

## Prompting

In [29]:
HUGGINGFACE_TOKEN = None # @TODO: Set your HuggingFace token here
import os
os.makedirs('./results_replication', exist_ok=True)
for model_name in ['meta-llama/Llama-2-7b-chat-hf', 'mistralai/Mistral-7B-Instruct-v0.1', 'lmsys/vicuna-7b-v1.5']:
    tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir='./model', token=HUGGINGFACE_TOKEN)
    
    # Generate prompts for politician simulation
    statements = []
    for index, row in df_2022.iterrows():
        statements.append(tokenizer(f'USER: Genera una declaración de {row["Name"]}, un parlamentario en Chile.\nASSISTANT: En 2019, {row["Name"]} dijo que', return_tensors="pt")['input_ids'])
    pickle.dump(statements, open(f'./results_replication/{model_name.replace("/", "_")}_politicians.pkl', 'wb'))
    
    # Generate prompts for news media simulation

## Extracting Activations

In [ ]:
HUGGINGFACE_TOKEN = None # @TODO: Set your HuggingFace token here
import os
os.makedirs('./results_replication', exist_ok=True)
def extract_attention_head_activations(model, statements):
    HEADS = [f"model.layers.{i}.self_attn.o_proj" for i in range(model.config.num_hidden_layers)]
    head_wise_hidden_states_list = []
    for prompt in tqdm(statements, total=len(statements)):
        with torch.no_grad():
            with TraceDict(model, HEADS) as ret:
                output = model(prompt.to('cuda'), output_hidden_states=True, output_attentions=True)
                head_wise_hidden_states = [ret[head].output.squeeze().detach().cpu() for head in HEADS]
                head_wise_hidden_states = torch.stack(head_wise_hidden_states, dim = 0).squeeze().numpy()
                head_wise_hidden_states_list.append(head_wise_hidden_states[:, :, :])
    features = []
    for head_wise_hidden_states, label in zip(head_wise_hidden_states_list, labels):
        features.append(rearrange([np.array(head_wise_hidden_states[:,-1,:])], 'b l (h d) -> b l h d', h = model.config.num_attention_heads))
    features = np.stack(features, axis=0)
    return features

for model_name in ['meta-llama/Llama-2-7b-chat-hf', 'mistralai/Mistral-7B-Instruct-v0.1', 'lmsys/vicuna-7b-v1.5']: #
    model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir='./model', low_cpu_mem_usage=True, torch_dtype=torch.float16, token=HUGGINGFACE_TOKEN).to('cuda:0')
    # Extract activations for politicians
    statements = pickle.load(open(f'./results_replication/{model_name.replace("/", "_")}_politicians.pkl', 'rb'))
    labels = np.array(df_2022['dim_ideology_P6'].astype(float))
    features = extract_attention_head_activations(model, statements)
    pickle.dump((features, labels), open(f"./results_replication/{model_name.replace('/','_')}_politician_features.pkl", 'wb'))

    # Extract activations for news media

## Probing

In [ ]:
import os
os.makedirs('./results_replication', exist_ok=True)
for model_name in ['meta-llama/Llama-2-7b-chat-hf', 'mistralai/Mistral-7B-Instruct-v0.1', 'lmsys/vicuna-7b-v1.5']:
    features, labels = pickle.load(open(f"./results_replication/{model_name.replace('/','_')}_politician_features.pkl", 'rb'))
    performance = np.zeros((model.config.num_hidden_layers, model.config.num_attention_heads))
    ridge_dict = {}
    for i in tqdm(range(model.config.num_hidden_layers)):
        ridge_dict[i] = {}
        for j in range(model.config.num_attention_heads):
            kf = KFold(n_splits=2, shuffle=True, random_state=42)
            for train_indices, test_indices in kf.split(range(features.shape[0])):
                X_train = features[train_indices, 0, i, j, :]
                X_test = features[test_indices, 0, i, j, :]
                y_train = np.array(labels)[train_indices]
                y_test = np.array(labels)[test_indices]
                ridge_model = Ridge(alpha=1, fit_intercept=False)
                ridge_model.fit(X_train, y_train)
                ridge_dict[i][j] = ridge_model
                y_pred = ridge_model.predict(X_test)
                performance[i, j] += spearmanr(y_test, y_pred).statistic
    performance /= 2
    pickle.dump(performance, open(f"./results_replication/{model_name.replace('/','_')}_politician_performance.pkl", 'wb'))
    pickle.dump(ridge_dict, open(f"./results_replication/{model_name.replace('/','_')}_ridge.pkl", 'wb'))

# Intervention

In [ ]:
HUGGINGFACE_TOKEN = None # @TODO: Set your HuggingFace token here
import os
os.makedirs('./results_replication', exist_ok=True)
def lt_modulated_vector_add(head_output, layer_name):
    layer_index = layer_name[len('model.layers.'):]
    layer_index = int(layer_index[:layer_index.index('.')])
    head_output = rearrange(head_output.detach().cpu(), 'b s (h d) -> b s h d', h=model.config.num_attention_heads)
    for head_index in head_dict[layer_index]:
        head_output[:, -1, head_index, :] += alpha  * focal_ridge_dict[(layer_index, head_index)] * np.std(features[:, 0, layer_index, head_index, :], axis=0)
    head_output = rearrange(head_output, 'b s h d -> b s (h d)')
    return head_output.to('cuda')

for model_name in ['meta-llama/Llama-2-7b-chat-hf', 'mistralai/Mistral-7B-Instruct-v0.1', 'lmsys/vicuna-7b-v1.5']:
    tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir='./model', token=HUGGINGFACE_TOKEN)
    model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir='./model', low_cpu_mem_usage=True, torch_dtype=torch.float16, 
                                                   token=HUGGINGFACE_TOKEN).to('cuda:0')
    performance = pickle.load(open(f"./results_replication/{model_name.replace('/','_')}_politician_performance.pkl", 'rb'))
    features, labels = pickle.load(open(f"./results_replication/{model_name.replace('/','_')}_politician_features.pkl", 'rb'))
    trained_ridge_dict = pickle.load(open(f"./results_replication/{model_name.replace('/','_')}_ridge.pkl", 'rb'))
    topics = ['aborto', 'inmigración', 'porte de armas', 'matrimonio igualitario', 'derechos LGBTQ+', 
              'cambio climático', 'confianza en ciencia', 'regulación de redes sociales', 'libertad de expresión']
    results = []
    for k in reversed([16, 32, 48, 64, 80, 96]):
        for alpha in tqdm(reversed([-30, -20, -10, 0, 10, 20, 30])): # Add -50, -40, 40, 50 for the coherence tests
            for topic in topics:
                print(k, alpha, topic)
                original_prompt = f"Escribe una declaración sobre {topic}."
                top_indices = np.dstack(np.unravel_index(np.argsort(performance.ravel()), (32, 32)))[0][-k:, :][::-1]
                focal_ridge_dict = {}
                for i in top_indices:
                    ridge_model = trained_ridge_dict[i[0]][i[1]]
                    focal_ridge_dict[tuple(i)] = ridge_model.coef_
                head_dict = {}
                for i in top_indices:
                    if i[0] not in head_dict:
                        head_dict[i[0]] = [i[1]]
                    else:
                        head_dict[i[0]].append(i[1])
                with TraceDict(model, [f'model.layers.{i}.self_attn.o_proj' for i in sorted(list(set(top_indices[:,0])))], edit_output=lt_modulated_vector_add) as ret: 
                    input_ids = tokenizer(f"USER: {original_prompt}\nASSISTANT: Respecto a {topic}, creo que ", return_tensors="pt")['input_ids']
                    model_gen_tokens = model.generate(input_ids.to('cuda')[0][:-1].unsqueeze(0), max_length=200)
                model_gen_str = tokenizer.decode(model_gen_tokens[0], skip_special_tokens=True)
                model_gen_str = model_gen_str.strip()
                results.append([k, alpha, topic, model_gen_str])
    pickle.dump(results, open(f"./results_replication/{model_name.replace('/','_')}_intervention_results.pkl", 'wb'))

In [ ]:
import os
os.makedirs('./results_replication', exist_ok=True)
results = []
for model_name in ['meta-llama/Llama-2-7b-chat-hf', 'mistralai/Mistral-7B-Instruct-v0.1', 'lmsys/vicuna-7b-v1.5']:
    results += pickle.load(open(f"./results_replication/{model_name.replace('/','_')}_intervention_results.pkl", 'rb'))

pd.DataFrame(results).to_parquet('intervention_raw.parquet')

In [2]:
import pandas as pd
pd1 = pd.read_csv('../data/congreso_2022_2026_con_resumen.csv')
pd2 = pd.read_csv('../data/congreso_2022_2026_biografias.csv')
pd_final = pd.merge(pd1, pd2[['Id', 'intervenciones']], on='Id', how='inner')
pd_final.head()
pd_final.to_csv('../data/congreso_2022_2026_con_resumen_e_intervenciones.csv', index=False)

In [6]:
prompts = []
for _, row in pd_final.iterrows():
    intervenciones = row.get('intervenciones')
    if pd.notna(intervenciones) and intervenciones:
        if isinstance(intervenciones, list):
            texto_intervenciones = "\n".join(intervenciones)
        else:
            texto_intervenciones = str(intervenciones)
        
        # Truncar para evitar OOM con prompts demasiado largos
        texto_intervenciones = texto_intervenciones[:3000]
        
        prompt_intervenciones = f"Además, considera las siguientes intervenciones parlamentarias:\n{texto_intervenciones}\n"
    else:
        prompt_intervenciones = ""
    prompt_text = f"""USER: Genera una declaración de {row["Name"]}, parlamentario/a en Chile del partido {row['party_1_P6']}.
    A continuación un resumen de su biografía:
    {row['resumen']}

    {prompt_intervenciones}"""
    prompts.append(prompt_text)

pd_final['prompt'] = pd.Series(prompts)
pd_final.head()
pd_final.to_csv('../data/congreso_2022_2026_con_resumen_e_intervenciones', index=False)